In [1]:
!pip install pandas nltk scikit-learn sacrebleu


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import json
import re
import ast
from typing import List, Dict, Optional, Tuple
from collections import Counter
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score
import numpy as np
import os

In [3]:
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("✓ Importazioni completate")

✓ Importazioni completate


In [4]:
# Configurazione
MODEL_NAME = "qwen3:4b-instruct"
MODEL_NAME_CLEAN = MODEL_NAME.replace(':', '_')
BASE_PATH = f"../Output Performance Degradation Analysis/{MODEL_NAME_CLEAN}_six_task"

# Task configurati - AGGIORNATO PER 6 TASK
TASK_NAMES = {
    1: "json_only",
    2: "json_translation", 
    3: "json_translation_sentiment",
    4: "json_translation_sentiment_emotion",
    5: "json_translation_sentiment_emotion_topic",
    6: "json_translation_sentiment_emotion_topic_ner"
}

# Campi richiesti per ogni task - AGGIORNATO PER 6 TASK
REQUIRED_FIELDS = {
    1: ["review"],
    2: ["review", "translate"],
    3: ["review", "translate", "sentiment"],
    4: ["review", "translate", "sentiment", "emotion"],
    5: ["review", "translate", "sentiment", "emotion", "topic"],
    6: ["review", "translate", "sentiment", "emotion", "topic", "entities"]
}

print(f"✓ Configurazione completata:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - Percorso base: {BASE_PATH}")
print(f"  - Task configurati: {len(TASK_NAMES)}")

✓ Configurazione completata:
  - Modello: qwen3:4b-instruct
  - Percorso base: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task
  - Task configurati: 6


In [5]:
def is_valid_json_format(json_string: str, required_fields: List[str]) -> bool:
    """
    Verifica se il JSON è formalmente valido e ha tutti i campi richiesti.
    
    Args:
        json_string (str): Il JSON da validare
        required_fields (List[str]): I campi richiesti
        
    Returns:
        bool: True se valido, False altrimenti
    """
    if pd.isna(json_string) or json_string is None:
        return False
    
    try:
        # Parse del JSON
        json_obj = json.loads(json_string)
        
        # Verifica che sia un dizionario
        if not isinstance(json_obj, dict):
            return False
        
        # Verifica che abbia tutti i campi richiesti
        json_fields = set(json_obj.keys())
        required_fields_set = set(required_fields)
        
        # Deve avere tutti i campi richiesti e nessun campo extra
        if json_fields != required_fields_set:
            return False
            
        return True
        
    except (json.JSONDecodeError, TypeError):
        return False

In [6]:
def calculate_bleu_score(reference: str, candidate: str) -> float:
    """
    Calcola il BLEU score tra reference e candidate.
    
    Args:
        reference (str): Testo di riferimento
        candidate (str): Testo candidato
        
    Returns:
        float: BLEU score (0-100)
    """
    if pd.isna(reference) or pd.isna(candidate) or reference == "" or candidate == "":
        return 0.0
    
    try:
        # Tokenizza i testi
        ref_tokens = nltk.word_tokenize(reference.lower())
        cand_tokens = nltk.word_tokenize(candidate.lower())
        
        # Calcola BLEU score con smoothing
        smoothing = SmoothingFunction()
        score = sentence_bleu([ref_tokens], cand_tokens, 
                            smoothing_function=smoothing.method1)
        
        return score * 100  # Converti in percentuale
        
    except Exception:
        return 0.0

In [12]:
def exact_match_classification(reference: str, candidate: str) -> bool:
    """
    Verifica exact match per classificazione (sentiment, emotion, topic).
    
    Args:
        reference (str): Valore di riferimento
        candidate (str): Valore candidato
        
    Returns:
        bool: True se match esatto, False altrimenti
    """
    # Gestisce il caso in cui i valori potrebbero essere None, NaN o array
    try:
        if reference is None or candidate is None:
            return False
        
        # Converte a stringa per sicurezza
        ref_str = str(reference)
        cand_str = str(candidate)
        
        # Controlla se sono valori NaN come stringa
        if ref_str.lower() in ['nan', 'none', ''] or cand_str.lower() in ['nan', 'none', '']:
            return False
        
        return ref_str.lower().strip() == cand_str.lower().strip()
        
    except Exception:
        # In caso di qualsiasi errore, restituisce False
        return False

In [13]:
def parse_entities(entities_str: str) -> List[Dict]:
    """
    Parsifica la stringa delle entità in una lista di dizionari.
    
    Args:
        entities_str (str): Stringa delle entità
        
    Returns:
        List[Dict]: Lista delle entità parsificate
    """
    if pd.isna(entities_str) or entities_str == "":
        return []
    
    try:
        # Prova a parsificare come JSON
        if entities_str.startswith('['):
            return json.loads(entities_str)
        
        # Prova a parsificare come literal Python
        return ast.literal_eval(entities_str)
        
    except (json.JSONDecodeError, ValueError, SyntaxError):
        return []

In [14]:
def calculate_ner_f1(reference_entities: str, candidate_entities: List[Dict]) -> float:
    """
    Calcola F1 score per NER.
    
    Args:
        reference_entities (str): Entità di riferimento
        candidate_entities (List[Dict]): Entità candidate
        
    Returns:
        float: F1 score (0-100)
    """
    # Parsifica le entità di riferimento
    ref_entities = parse_entities(reference_entities)
    
    if not ref_entities and not candidate_entities:
        return 100.0  # Entrambi vuoti = match perfetto
    
    if not ref_entities or not candidate_entities:
        return 0.0  # Uno vuoto, uno no = no match
    
    # Estrai set di (label, value) per confronto
    ref_set = set()
    for entity in ref_entities:
        if isinstance(entity, dict) and 'label' in entity and 'value' in entity:
            ref_set.add((entity['label'], entity['value'].lower().strip()))
    
    cand_set = set()
    for entity in candidate_entities:
        if isinstance(entity, dict) and 'label' in entity and 'value' in entity:
            cand_set.add((entity['label'], entity['value'].lower().strip()))
    
    if not ref_set and not cand_set:
        return 100.0
    
    if not ref_set or not cand_set:
        return 0.0
    
    # Calcola precisione, recall, F1
    intersection = ref_set.intersection(cand_set)
    
    if len(intersection) == 0:
        return 0.0
    
    precision = len(intersection) / len(cand_set)
    recall = len(intersection) / len(ref_set)
    
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1 * 100  # Converti in percentuale

In [15]:
def evaluate_task_file(task_number: int) -> pd.DataFrame:
    """
    Valuta un singolo file di task e calcola tutte le metriche.
    
    Args:
        task_number (int): Numero del task (1-6)
        
    Returns:
        pd.DataFrame: DataFrame con tutte le metriche calcolate
    """
    task_name = TASK_NAMES[task_number]
    required_fields = REQUIRED_FIELDS[task_number]
    
    # Percorso file
    input_file = f"{BASE_PATH}_cleaned/sampled_reviews_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}_cleaned.csv"
    
    print(f"Valutando Task {task_number} ({task_name})...")
    
    # Verifica esistenza file
    if not os.path.exists(input_file):
        print(f"❌ File non trovato: {input_file}")
        return pd.DataFrame()
    
    # Carica il dataframe
    df = pd.read_csv(input_file)
    print(f"✓ File caricato: {len(df)} righe")
    
    # Inizializza colonne metriche
    df['json_valid'] = 0
    df['bleu_translate'] = 0.0
    df['em_sentiment'] = 0
    df['em_emotion'] = 0
    df['em_topic'] = 0
    df['f1_ner'] = 0.0
    
    # Calcola metriche per ogni riga
    for idx, row in df.iterrows():
        json_output = row.get('json_output', None)
        
        # 1. Validazione JSON
        df.at[idx, 'json_valid'] = 1 if is_valid_json_format(json_output, required_fields) else 0
        
        # Se JSON non è valido, salta le altre metriche
        if df.at[idx, 'json_valid'] == 0:
            continue
        
        try:
            json_obj = json.loads(json_output)
            
            # 2. BLEU su translate (se richiesto)
            if 'translate' in required_fields:
                ref_translate = row.get('translate', '')
                cand_translate = json_obj.get('translate', '')
                df.at[idx, 'bleu_translate'] = calculate_bleu_score(ref_translate, cand_translate)
            
            # 3. EM su sentiment (se richiesto)
            if 'sentiment' in required_fields:
                ref_sentiment = row.get('sentiment', '')
                cand_sentiment = json_obj.get('sentiment', '')
                df.at[idx, 'em_sentiment'] = 1 if exact_match_classification(ref_sentiment, cand_sentiment) else 0
            
            # 4. EM su emotion (se richiesto)
            if 'emotion' in required_fields:
                ref_emotion = row.get('emotion_classification', '')
                cand_emotion = json_obj.get('emotion', '')
                df.at[idx, 'em_emotion'] = 1 if exact_match_classification(ref_emotion, cand_emotion) else 0
            
            # 5. EM su topic (se richiesto)
            if 'topic' in required_fields:
                ref_topic = row.get('topic_classification', '')
                cand_topic = json_obj.get('topic', '')
                df.at[idx, 'em_topic'] = 1 if exact_match_classification(ref_topic, cand_topic) else 0
            
            # 6. F1 su NER (se richiesto)
            if 'entities' in required_fields:
                ref_entities = row.get('entities', '')
                cand_entities = json_obj.get('entities', [])
                df.at[idx, 'f1_ner'] = calculate_ner_f1(ref_entities, cand_entities)
                
        except (json.JSONDecodeError, TypeError):
            # JSON non valido, metriche già a 0
            pass
    
    return df

In [16]:
print("=" * 60)
print("CARICAMENTO E VALUTAZIONE TASK")
print("=" * 60)

# Dizionario per memorizzare i risultati
task_results = {}

for task_number in TASK_NAMES.keys():
    task_results[task_number] = evaluate_task_file(task_number)
    print("-" * 40)

print("✅ Valutazione completata per tutti i task!")

CARICAMENTO E VALUTAZIONE TASK
Valutando Task 1 (json_only)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 2 (json_translation)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 3 (json_translation_sentiment)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 4 (json_translation_sentiment_emotion)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 5 (json_translation_sentiment_emotion_topic)...
✓ File caricato: 500 righe
----------------------------------------
Valutando Task 6 (json_translation_sentiment_emotion_topic_ner)...
✓ File caricato: 500 righe
----------------------------------------
✅ Valutazione completata per tutti i task!


In [17]:
print("=" * 60)
print("STEP 1: EXACT MATCH SU JSON")
print("=" * 60)

step1_results = {}

for task_number in TASK_NAMES.keys():
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Calcola metriche Step 1
    json_valid_mean = df['json_valid'].mean() * 100
    
    step1_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 1:")
for task_num, results in step1_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%")

STEP 1: EXACT MATCH SU JSON
Task 1 (json_only):
  - JSON Exact Match: 99.00%
  - Campioni totali: 500
Task 2 (json_translation):
  - JSON Exact Match: 96.40%
  - Campioni totali: 500
Task 3 (json_translation_sentiment):
  - JSON Exact Match: 95.60%
  - Campioni totali: 500
Task 4 (json_translation_sentiment_emotion):
  - JSON Exact Match: 96.40%
  - Campioni totali: 500
Task 5 (json_translation_sentiment_emotion_topic):
  - JSON Exact Match: 96.40%
  - Campioni totali: 500
Task 6 (json_translation_sentiment_emotion_topic_ner):
  - JSON Exact Match: 96.40%
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 1:
Task 1: JSON EM = 99.00%
Task 2: JSON EM = 96.40%
Task 3: JSON EM = 95.60%
Task 4: JSON EM = 96.40%
Task 5: JSON EM = 96.40%
Task 6: JSON EM = 96.40%


In [18]:
print("=" * 60)
print("STEP 2: EM JSON + BLEU TRANSLATE")
print("=" * 60)

step2_results = {}

for task_number in [2, 3, 4, 5, 6]:  # Task con translate
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 2
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    
    step2_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 2:")
for task_num, results in step2_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%")


STEP 2: EM JSON + BLEU TRANSLATE
Task 2 (json_translation):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.73% (su 482 JSON validi)
  - Campioni totali: 500
Task 3 (json_translation_sentiment):
  - JSON Exact Match: 95.60%
  - BLEU Translate: 38.49% (su 478 JSON validi)
  - Campioni totali: 500
Task 4 (json_translation_sentiment_emotion):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.59% (su 482 JSON validi)
  - Campioni totali: 500
Task 5 (json_translation_sentiment_emotion_topic):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.27% (su 482 JSON validi)
  - Campioni totali: 500
Task 6 (json_translation_sentiment_emotion_topic_ner):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.09% (su 482 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 2:
Task 2: JSON EM = 96.40%, BLEU = 38.73%
Task 3: JSON EM = 95.60%, BLEU = 38.49%
Task 4: JSON EM = 96.40%, BLEU = 38.59%
Task 5: JSON EM = 96.40%, BLEU = 38.27%
Tas

In [19]:
print("=" * 60)
print("STEP 3: EM JSON + BLEU TRANSLATE + EM SENTIMENT")
print("=" * 60)

step3_results = {}

for task_number in [3, 4, 5, 6]:  # Task con sentiment
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 3
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    
    step3_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'em_sentiment': em_sentiment_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 3:")
for task_num, results in step3_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%")


STEP 3: EM JSON + BLEU TRANSLATE + EM SENTIMENT
Task 3 (json_translation_sentiment):
  - JSON Exact Match: 95.60%
  - BLEU Translate: 38.49% (su 478 JSON validi)
  - EM Sentiment: 92.47% (su 478 JSON validi)
  - Campioni totali: 500
Task 4 (json_translation_sentiment_emotion):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.59% (su 482 JSON validi)
  - EM Sentiment: 90.66% (su 482 JSON validi)
  - Campioni totali: 500
Task 5 (json_translation_sentiment_emotion_topic):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.27% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - Campioni totali: 500
Task 6 (json_translation_sentiment_emotion_topic_ner):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.09% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 3:
Task 3: JSON EM = 95.60%, BLEU = 38.49%, Sentiment EM = 92.47%
Task 4: JSON EM = 96.40%, BLEU

In [20]:
print("=" * 60)
print("STEP 4: EM JSON + BLEU TRANSLATE + EM SENTIMENT + EM EMOTION")
print("=" * 60)

step4_results = {}

for task_number in [4, 5, 6]:  # Task con emotion
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 4
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    
    step4_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 4:")
for task_num, results in step4_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%")


STEP 4: EM JSON + BLEU TRANSLATE + EM SENTIMENT + EM EMOTION
Task 4 (json_translation_sentiment_emotion):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.59% (su 482 JSON validi)
  - EM Sentiment: 90.66% (su 482 JSON validi)
  - EM Emotion: 45.85% (su 482 JSON validi)
  - Campioni totali: 500
Task 5 (json_translation_sentiment_emotion_topic):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.27% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - EM Emotion: 45.64% (su 482 JSON validi)
  - Campioni totali: 500
Task 6 (json_translation_sentiment_emotion_topic_ner):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.09% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - EM Emotion: 42.53% (su 482 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 4:
Task 4: JSON EM = 96.40%, BLEU = 38.59%, Sentiment EM = 90.66%, Emotion EM = 45.85%
Task 5: JSON EM = 96.40%, BLEU = 38.27%, Sentimen

In [21]:
print("=" * 60)
print("STEP 5: EM JSON + BLEU TRANSLATE + EM SENTIMENT + EM EMOTION + EM TOPIC")
print("=" * 60)

step5_results = {}

for task_number in [5, 6]:  # Task con topic
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 5
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    em_topic_mean = valid_json_df['em_topic'].mean() * 100
    
    step5_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'em_topic': em_topic_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Topic: {em_topic_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 5:")
for task_num, results in step5_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%, Topic EM = {results['em_topic']:.2f}%")


STEP 5: EM JSON + BLEU TRANSLATE + EM SENTIMENT + EM EMOTION + EM TOPIC
Task 5 (json_translation_sentiment_emotion_topic):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.27% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - EM Emotion: 45.64% (su 482 JSON validi)
  - EM Topic: 61.20% (su 482 JSON validi)
  - Campioni totali: 500
Task 6 (json_translation_sentiment_emotion_topic_ner):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.09% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - EM Emotion: 42.53% (su 482 JSON validi)
  - EM Topic: 57.47% (su 482 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 5:
Task 5: JSON EM = 96.40%, BLEU = 38.27%, Sentiment EM = 91.70%, Emotion EM = 45.64%, Topic EM = 61.20%
Task 6: JSON EM = 96.40%, BLEU = 38.09%, Sentiment EM = 91.70%, Emotion EM = 42.53%, Topic EM = 57.47%


In [22]:
print("=" * 60)
print("STEP 6: TUTTE LE METRICHE")
print("=" * 60)

step6_results = {}

for task_number in [6]:  # Solo task con NER
    df = task_results[task_number]
    if df.empty:
        continue
    
    task_name = TASK_NAMES[task_number]
    
    # Filtra solo righe con JSON valido
    valid_json_df = df[df['json_valid'] == 1]
    
    if len(valid_json_df) == 0:
        print(f"Task {task_number} ({task_name}): Nessun JSON valido trovato")
        continue
    
    # Calcola metriche Step 6
    json_valid_mean = df['json_valid'].mean() * 100
    bleu_mean = valid_json_df['bleu_translate'].mean()
    em_sentiment_mean = valid_json_df['em_sentiment'].mean() * 100
    em_emotion_mean = valid_json_df['em_emotion'].mean() * 100
    em_topic_mean = valid_json_df['em_topic'].mean() * 100
    f1_ner_mean = valid_json_df['f1_ner'].mean()
    
    step6_results[task_number] = {
        'task_name': task_name,
        'json_em': json_valid_mean,
        'bleu_translate': bleu_mean,
        'em_sentiment': em_sentiment_mean,
        'em_emotion': em_emotion_mean,
        'em_topic': em_topic_mean,
        'f1_ner': f1_ner_mean,
        'valid_json_count': len(valid_json_df),
        'total_samples': len(df)
    }
    
    print(f"Task {task_number} ({task_name}):")
    print(f"  - JSON Exact Match: {json_valid_mean:.2f}%")
    print(f"  - BLEU Translate: {bleu_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Sentiment: {em_sentiment_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Emotion: {em_emotion_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - EM Topic: {em_topic_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - F1 NER: {f1_ner_mean:.2f}% (su {len(valid_json_df)} JSON validi)")
    print(f"  - Campioni totali: {len(df)}")

print("-" * 60)
print("RIEPILOGO STEP 6:")
for task_num, results in step6_results.items():
    print(f"Task {task_num}: JSON EM = {results['json_em']:.2f}%, BLEU = {results['bleu_translate']:.2f}%, Sentiment EM = {results['em_sentiment']:.2f}%, Emotion EM = {results['em_emotion']:.2f}%, Topic EM = {results['em_topic']:.2f}%, NER F1 = {results['f1_ner']:.2f}%")


STEP 6: TUTTE LE METRICHE
Task 6 (json_translation_sentiment_emotion_topic_ner):
  - JSON Exact Match: 96.40%
  - BLEU Translate: 38.09% (su 482 JSON validi)
  - EM Sentiment: 91.70% (su 482 JSON validi)
  - EM Emotion: 42.53% (su 482 JSON validi)
  - EM Topic: 57.47% (su 482 JSON validi)
  - F1 NER: 66.54% (su 482 JSON validi)
  - Campioni totali: 500
------------------------------------------------------------
RIEPILOGO STEP 6:
Task 6: JSON EM = 96.40%, BLEU = 38.09%, Sentiment EM = 91.70%, Emotion EM = 42.53%, Topic EM = 57.47%, NER F1 = 66.54%


In [25]:
print("=" * 80)
print("ANALISI DELLA DEGRADAZIONE DELLE PERFORMANCE")
print("=" * 80)

# Analisi JSON EM attraverso tutti gli step
print("DEGRADAZIONE JSON EXACT MATCH:")
print("-" * 50)

json_em_progression = {}
for task_num in [1, 2, 3, 4, 5, 6]:
    json_values = []
    
    if task_num in step1_results:
        json_values.append(step1_results[task_num]['json_em'])
    if task_num in step2_results:
        json_values.append(step2_results[task_num]['json_em'])
    if task_num in step3_results:
        json_values.append(step3_results[task_num]['json_em'])
    if task_num in step4_results:
        json_values.append(step4_results[task_num]['json_em'])
    if task_num in step5_results:
        json_values.append(step5_results[task_num]['json_em'])
    if task_num in step6_results:
        json_values.append(step6_results[task_num]['json_em'])
    
    if json_values:
        json_em_progression[task_num] = json_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in json_values])}")

# Analisi BLEU attraverso gli step dove è applicabile
print(f"\nDEGRADAZIONE BLEU TRANSLATE:")
print("-" * 50)

bleu_progression = {}
for task_num in [2, 3, 4, 5, 6]:
    bleu_values = []
    
    if task_num in step2_results:
        bleu_values.append(step2_results[task_num]['bleu_translate'])
    if task_num in step3_results:
        bleu_values.append(step3_results[task_num]['bleu_translate'])
    if task_num in step4_results:
        bleu_values.append(step4_results[task_num]['bleu_translate'])
    if task_num in step5_results:
        bleu_values.append(step5_results[task_num]['bleu_translate'])
    if task_num in step6_results:
        bleu_values.append(step6_results[task_num]['bleu_translate'])
    
    if bleu_values:
        bleu_progression[task_num] = bleu_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in bleu_values])}")

# Analisi Sentiment EM
print(f"\nDEGRADAZIONE SENTIMENT EXACT MATCH:")
print("-" * 50)

sentiment_progression = {}
for task_num in [3, 4, 5, 6]:
    sentiment_values = []
    
    if task_num in step3_results:
        sentiment_values.append(step3_results[task_num]['em_sentiment'])
    if task_num in step4_results:
        sentiment_values.append(step4_results[task_num]['em_sentiment'])
    if task_num in step5_results:
        sentiment_values.append(step5_results[task_num]['em_sentiment'])
    if task_num in step6_results:
        sentiment_values.append(step6_results[task_num]['em_sentiment'])
    
    if sentiment_values:
        sentiment_progression[task_num] = sentiment_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in sentiment_values])}")

# Analisi Emotion EM
print(f"\nDEGRADAZIONE EMOTION EXACT MATCH:")
print("-" * 50)

emotion_progression = {}
for task_num in [4, 5, 6]:
    emotion_values = []
    
    if task_num in step4_results:
        emotion_values.append(step4_results[task_num]['em_emotion'])
    if task_num in step5_results:
        emotion_values.append(step5_results[task_num]['em_emotion'])
    if task_num in step6_results:
        emotion_values.append(step6_results[task_num]['em_emotion'])
    
    if emotion_values:
        emotion_progression[task_num] = emotion_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in emotion_values])}")

# Analisi Topic EM
print(f"\nDEGRADAZIONE TOPIC EXACT MATCH:")
print("-" * 50)

topic_progression = {}
for task_num in [5, 6]:
    topic_values = []
    
    if task_num in step5_results:
        topic_values.append(step5_results[task_num]['em_topic'])
    if task_num in step6_results:
        topic_values.append(step6_results[task_num]['em_topic'])
    
    if topic_values:
        topic_progression[task_num] = topic_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in topic_values])}")

# Analisi NER F1
print(f"\nDEGRADAZIONE NER F1:")
print("-" * 50)

ner_progression = {}
for task_num in [6]:
    ner_values = []
    
    if task_num in step6_results:
        ner_values.append(step6_results[task_num]['f1_ner'])
    
    if ner_values:
        ner_progression[task_num] = ner_values
        print(f"Task {task_num}: {' -> '.join([f'{v:.2f}%' for v in ner_values])}")

print("\n✅ Analisi della degradazione completata!")

ANALISI DELLA DEGRADAZIONE DELLE PERFORMANCE
DEGRADAZIONE JSON EXACT MATCH:
--------------------------------------------------
Task 1: 99.00%
Task 2: 96.40% -> 96.40%
Task 3: 95.60% -> 95.60% -> 95.60%
Task 4: 96.40% -> 96.40% -> 96.40% -> 96.40%
Task 5: 96.40% -> 96.40% -> 96.40% -> 96.40% -> 96.40%
Task 6: 96.40% -> 96.40% -> 96.40% -> 96.40% -> 96.40% -> 96.40%

DEGRADAZIONE BLEU TRANSLATE:
--------------------------------------------------
Task 2: 38.73%
Task 3: 38.49% -> 38.49%
Task 4: 38.59% -> 38.59% -> 38.59%
Task 5: 38.27% -> 38.27% -> 38.27% -> 38.27%
Task 6: 38.09% -> 38.09% -> 38.09% -> 38.09% -> 38.09%

DEGRADAZIONE SENTIMENT EXACT MATCH:
--------------------------------------------------
Task 3: 92.47%
Task 4: 90.66% -> 90.66%
Task 5: 91.70% -> 91.70% -> 91.70%
Task 6: 91.70% -> 91.70% -> 91.70% -> 91.70%

DEGRADAZIONE EMOTION EXACT MATCH:
--------------------------------------------------
Task 4: 45.85%
Task 5: 45.64% -> 45.64%
Task 6: 42.53% -> 42.53% -> 42.53%

DEGRADA

In [28]:
print("=" * 60)
print("SALVATAGGIO RISULTATI")
print("=" * 60)

# Crea un DataFrame riassuntivo
summary_data = []

# Aggiungi risultati di tutti gli step
for step_name, step_results in [
    ("Step 1", step1_results),
    ("Step 2", step2_results), 
    ("Step 3", step3_results),
    ("Step 4", step4_results),
    ("Step 5", step5_results),
    ("Step 6", step6_results)
]:
    for task_num, results in step_results.items():
        row = {
            'step': step_name,
            'task_number': task_num,
            'task_name': results['task_name'],
            'json_em': results['json_em'],
            'bleu_translate': results.get('bleu_translate', None),
            'em_sentiment': results.get('em_sentiment', None),
            'em_emotion': results.get('em_emotion', None),
            'em_topic': results.get('em_topic', None),
            'f1_ner': results.get('f1_ner', None),
            'total_samples': results['total_samples'],
            'valid_json_count': results.get('valid_json_count', None)
        }
        summary_data.append(row)

# Crea DataFrame riassuntivo
summary_df = pd.DataFrame(summary_data)

# Salva i risultati
output_file = f"{BASE_PATH}/evaluation_summary_{MODEL_NAME_CLEAN}_6tasks.csv"
summary_df.to_csv(output_file, index=False)

print(f"✓ Risultati salvati in: {output_file}")

# Mostra tabella riassuntiva
print("\n📋 TABELLA RIASSUNTIVA:")
print(summary_df.to_string(index=False))

print("\n✅ Script di valutazione completato!")

SALVATAGGIO RISULTATI
✓ Risultati salvati in: ../Output Performance Degradation Analysis/qwen3_4b-instruct_six_task/evaluation_summary_qwen3_4b-instruct_6tasks.csv

📋 TABELLA RIASSUNTIVA:
  step  task_number                                    task_name  json_em  bleu_translate  em_sentiment  em_emotion  em_topic    f1_ner  total_samples  valid_json_count
Step 1            1                                    json_only     99.0             NaN           NaN         NaN       NaN       NaN            500               NaN
Step 1            2                             json_translation     96.4             NaN           NaN         NaN       NaN       NaN            500               NaN
Step 1            3                   json_translation_sentiment     95.6             NaN           NaN         NaN       NaN       NaN            500               NaN
Step 1            4           json_translation_sentiment_emotion     96.4             NaN           NaN         NaN       NaN       NaN 